# 中断与继续

- `abort()`：设置当前 run 的中断信号，底层 stream 会尽快以 `stop_reason="aborted"` 结束。
- `continue_()`：从当前上下文继续。若最后一条是 assistant 消息，会先尝试 drain steering / follow-up 队列。


In [1]:
import os

os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"


In [2]:
import asyncio
from nova_agent import Agent
from nova_ai import UserMessage

agent = Agent(
    initial_state={"system_prompt": "你是长文本生成助手。"},
)

agent.subscribe(lambda e: print(f"[{e.type}]"))

# 1 秒后中断
async def interrupt():
    await asyncio.sleep(1.0)
    print("\n[调用 abort()]")
    agent.abort()

asyncio.create_task(interrupt())
await agent.prompt("请写一首很长的诗。")
await agent.wait_for_idle()

last = agent.state.messages[-1]
print("\n最后消息 stop_reason:", getattr(last, "stop_reason", None))
print("错误信息:", getattr(last, "error_message", None))


[agent_start]
[turn_start]
[message_start]
[message_end]

[调用 abort()]
[message_start]
[message_end]
[turn_end]
[agent_end]

最后消息 stop_reason: aborted
错误信息: Request was aborted


## 从中断后继续

中断后可以通过 `follow_up()` 注入新的用户消息，然后 `continue_()` 继续对话。


In [3]:
agent.follow_up(UserMessage(role="user", content="请继续完成刚才的诗。"))
await agent.continue_()
await agent.wait_for_idle()

print("\n最终回复:", agent.state.messages[-1].content[0].text[:200])


[agent_start]
[turn_start]
[message_start]
[message_end]
[message_start]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_